# 06 — Preparação de USD/BRL e Brent

Objetivo: padronizar as séries diárias e agregá-las para semanas de segunda a domingo, usando média, mínimo, máximo, último valor, variação semanal e volatilidade diária. Nenhum dado posterior à semana será deslocado para trás.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.external_data import (
    aggregate_daily_to_weekly, find_usd_files, load_brent_daily,
    load_usd_daily, save_external_data, weekly_coverage,
)

## 1. USD/BRL diário

In [ ]:
usd_files = find_usd_files(PROJECT_ROOT / 'raw')
usd_daily, usd_quality = load_usd_daily(usd_files)
display(pd.Series(usd_quality, name='valor').to_frame())
display(usd_daily.head())
display(usd_daily.tail())

## 2. Brent diário — EIA RBRTE

In [ ]:
brent_path = PROJECT_ROOT / 'raw' / 'eia' / 'brent' / 'brent_daily.xlsx.xls'
brent_daily, brent_quality = load_brent_daily(brent_path)
display(pd.Series(brent_quality, name='valor').to_frame())
display(brent_daily.head())
display(brent_daily.tail())

## 3. Agregação semanal

In [ ]:
usd_weekly = aggregate_daily_to_weekly(usd_daily, 'usd_brl', 'usd_brl')
brent_weekly = aggregate_daily_to_weekly(brent_daily, 'brent_usd_barril', 'brent')
display(usd_weekly.tail())
display(brent_weekly.tail())

In [ ]:
assert usd_weekly['semana_fim'].dt.dayofweek.eq(6).all()
assert brent_weekly['semana_fim'].dt.dayofweek.eq(6).all()
assert not usd_weekly['semana_fim'].duplicated().any()
assert not brent_weekly['semana_fim'].duplicated().any()
usd_required = ['usd_brl_media', 'usd_brl_minimo', 'usd_brl_maximo', 'usd_brl_ultimo', 'usd_brl_observacoes']
brent_required = ['brent_media', 'brent_minimo', 'brent_maximo', 'brent_ultimo', 'brent_observacoes']
assert usd_weekly[usd_required].notna().all().all()
assert brent_weekly[brent_required].notna().all().all()
print('Agregações semanais válidas.')

## 4. Cobertura nas semanas do alvo

In [ ]:
target = pd.read_csv(
    PROJECT_ROOT / 'processed' / 'anp' / 'revenda' / 'diesel_s10_rs_semanal.csv',
    parse_dates=['semana_fim'],
)
last_complete_week = pd.Timestamp('2026-06-28')
target_weeks = target.loc[target['semana_fim'].le(last_complete_week), 'semana_fim']
coverage = {
    'usd_brl': weekly_coverage(target_weeks, usd_weekly),
    'brent': weekly_coverage(target_weeks, brent_weekly),
}
pd.DataFrame(coverage).T

In [ ]:
assert coverage['usd_brl']['semanas_ausentes'] == 0
assert coverage['brent']['semanas_ausentes'] == 0
print('USD/BRL e Brent cobrem todas as semanas completas do alvo.')

## 5. Visualização no período comum

In [ ]:
common = (
    target[['semana_fim', 'preco_medio']]
    .merge(usd_weekly[['semana_fim', 'usd_brl_media']], on='semana_fim', how='inner')
    .merge(brent_weekly[['semana_fim', 'brent_media']], on='semana_fim', how='inner')
    .loc[lambda frame: frame['semana_fim'].le(last_complete_week)]
)
normalized = common.set_index('semana_fim').apply(lambda series: series / series.iloc[0] * 100)
ax = normalized.plot(figsize=(13, 5), title='Séries normalizadas — primeira semana = 100')
ax.set(xlabel='Semana encerrada em', ylabel='Índice')
ax.grid(alpha=0.25)
plt.show()

O gráfico é apenas exploratório. Movimentos contemporâneos não demonstram causalidade e ainda testaremos defasagens.

## 6. Salvar séries processadas

In [ ]:
quality = {'usd_brl': usd_quality, 'brent': brent_quality, 'cobertura_alvo': coverage}
output_paths = save_external_data(
    usd_daily, usd_weekly, brent_daily, brent_weekly, quality,
    PROJECT_ROOT / 'processed' / 'external',
)
output_paths

## Próximo passo

Construir o Dataset V1 juntando as variáveis semanais externas às features do Diesel. Serão criados lags para verificar possíveis defasagens sem usar valores futuros.